# 07. Tinier-YOLO USB-kamerával — felvétel kérésre

Használjon **közvetlenül a PYNQ-kártyához csatlakoztatott USB-kamerát** (például Logitech C270), ne a laptop webkameráját. A notebook az [05. laboratórium](05_tiny-yolo-image.ipynb) telepített Tinier-YOLO modelljét használja, 20 VOC-osztállyal, köztük személy, autó, kerékpár, kutya, palack és szék.

![Tinier-YOLO topológia](../Tinier-YOLO-topology.svg)

A cellákat sorrendben, **Shift + Enter** billentyűkkel futtassa; előbb állítsa le a többi FPGA- vagy kamerakernelt. A telepített `qnn` csomagból a nem használt `Xlnk` importokat már el kellett távolítani. Nem szükséges új modell, bitstream-fordítás, újratelepítés vagy tanítás.

Minden gombnyomás **új állóképet** rögzít, majd a kamerát felszabadítja és feldolgozza a képet. Nincs folyamatos háttér-kameraszál. A CPU-konvolúciók sokáig tarthatnak, ezért ez nem folyamatos élő videós bemutató. A korábbi eredmények láthatók maradnak, külön fájlokba nem kerülnek mentésre.

**Tárolás:** nincs eredménykönyvtár, PNG, pontszámfájl vagy JSON-napló. A natív könyvtár kisméretű ideiglenes fájlokat használ a `/dev/shm` (RAM) területén, amelyek feldolgozás és munkamenet-végi tisztítás után törlődnek. A csökkentett méretű előnézetek a kernel újraindításáig a memóriában maradnak. A Jupyter mentéskor a notebookba ágyazhatja a megjelenített képeket; mentés előtt használja a **Cell → All Output → Clear** parancsot. Az eredeti modellfájlok változatlanok.

*Az oktatási laboratórium fordítása; az alkönyvtárból való használathoz a szülőkönyvtár erőforrásait is ellenőrzi. A fordított példányt még nem validáltuk külön hardveren.*

## 1. Csomagok, modell és kamerabeállítások

Futtassa ezt a cellát. Megkeresi a meglévő modellt és az egyedi Darknet-telepítést, valamint felsorolja a `/dev/video*` eszközöket. Az alapértelmezett kamera `/dev/video0`, a kért felbontás 640×480. Másik kamera esetén ellenőrizze a listát, és módosítsa a `CAMERA_DEVICE` változót. Ha a kamera más felbontást választ, a tényleges képméretet használjuk.

Az FPGA inicializálása előtt a program jelzi a hiányzó fájlokat. Az `ipywidgets` biztosítja a gombokat, ha elérhető; a függvények kézzel is meghívhatók. A natív OpenCV/Darknet csomagok megegyeznek a kártyán már használt verziókkal.

In [ ]:
import sys
import os
import json
import io
import traceback
from collections import Counter
from contextlib import nullcontext
import cv2
import gc
import random
import time
import ctypes
import tempfile
import os
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from matplotlib import pyplot as plt
from IPython.display import display
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

import qnn
from qnn import TinierYolo, utils

QNN_PARAM_DIR = Path(qnn.__file__).resolve().parent / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "Tinier-YOLO-topology.svg").is_file():
    if (NOTEBOOK_DIR.parent / "Tinier-YOLO-topology.svg").is_file():
        NOTEBOOK_DIR = NOTEBOOK_DIR.parent
    else:
        NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
IMAGE_DIR = NOTEBOOK_DIR / "yoloimages"
DARKNET_DIR = Path("/opt/darknet")
VOC_NAMES = DARKNET_DIR / "data/voc.names"
required_files = [QNN_PARAM_DIR / name for name in (
    "tinier-yolo-layers.json", "tinier-yolo-conv0-W.npy", "tinier-yolo-conv8-W.npy",
    "tinier-yolo-conv0-bias.npy", "tinier-yolo-conv8-bias.npy",
    "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg")]
required_files += [DARKNET_DIR / "python/darknet.py", DARKNET_DIR / "libdarknet.so", VOC_NAMES]
# Darknet loads all printable-character PNGs at each size when drawing labels.
required_files += [DARKNET_DIR / "data/labels" / "{}_{}.png".format(char, size)
                   for size in range(8) for char in range(32, 127)]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing (first 12):\n" + "\n".join(missing[:12]))
if str(DARKNET_DIR / "python") not in sys.path:
    sys.path.insert(0, str(DARKNET_DIR / "python"))
import darknet as dn

dn.lib.parse_network_cfg.argtypes = [ctypes.c_char_p]
dn.lib.parse_network_cfg.restype = ctypes.c_void_p
dn.lib.forward_region_layer_pointer_nolayer.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_float)]
dn.lib.forward_region_layer_pointer_nolayer.restype = None
dn.lib.draw_detection_python.argtypes = [ctypes.c_void_p, ctypes.c_char_p,
    ctypes.c_float, ctypes.c_float, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p]
dn.lib.draw_detection_python.restype = None
dn.lib.free_network.argtypes = [ctypes.c_void_p]
dn.lib.free_network.restype = None
print("Parameters:", QNN_PARAM_DIR)
print("USB devices:", ", ".join(str(p) for p in sorted(Path("/dev").glob("video*"))) or "none")

CAMERA_DEVICE = "/dev/video0"
CAMERA_WIDTH, CAMERA_HEIGHT = 640, 480
WARMUP_FRAMES = 12
THRESHOLD = 0.3
CAPTURE_INDEX = 0
# Close controls from an earlier run before replacing model state.
for control in globals().get("ui_buttons", []):
    control.disabled = True
if globals().get("ui_panel") is not None:
    ui_panel.close()
print("Widgets:", "available" if widgets is not None else "not installed; use the manual call below")

def memory_preview(value):
    if isinstance(value, Image.Image):
        image = value.copy()
    else:
        with Image.open(str(value)) as opened:
            image = opened.convert("RGB")
    image.thumbnail((480, 360))
    return image

def clear_intermediate_files():
    directory = globals().get("RESULT_DIR")
    if directory is not None and directory.is_dir():
        for path in directory.iterdir():
            if path.is_file():
                path.unlink()



## 2. A modell egyszeri betöltése

Várja meg a **Model ready** üzenetet. Az első és utolsó konvolúció ARM CPU-n, a köztes kvantált rétegek FPGA-n futnak, a Darknet végzi a letterbox-átalakítást. A nagy súlyfájlok nem töltődnek le újra. A szakasz újrafuttatása új memóriabeli előzményeket indít; előtte zárja be a régi vezérlőket a **Finish / release** gombbal.

In [ ]:
def release_model():
    global classifier, net_darknet
    try:
        if globals().get("classifier") is not None:
            classifier.deinit_accelerator()
    finally:
        classifier = None
        if globals().get("net_darknet"):
            dn.lib.free_network(net_darknet)
        net_darknet = None
        if globals().get("_result_workspace") is not None:
            _result_workspace.cleanup()

release_model()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Loading weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv8_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv0_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
conv8_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
classifier = None
net_darknet = None
if globals().get("_result_workspace") is not None:
    _result_workspace.cleanup()
if not Path("/dev/shm").is_dir():
    raise RuntimeError("RAM-backed /dev/shm is required; no SD-card fallback is used.")
_result_workspace = tempfile.TemporaryDirectory(prefix="qnn-07-camera-", dir="/dev/shm")
RESULT_DIR = Path(_result_workspace.name)
try:
    classifier = TinierYolo()
    net = classifier.load_network()
    out_ch, out_dim, _ = net["conv7"]["output"]
    first_ch, first_dim, _ = net["conv1"]["input"]
    if conv0_weights.shape[0] != first_ch or conv0_bias.shape != (first_ch, 1):
        raise ValueError("First-convolution weights do not match the network.")
    if conv8_weights.shape[:2] != (125, out_ch) or conv8_bias.shape != (125, 1):
        raise ValueError("Expected the supplied 20-class, 5-anchor VOC detection head.")
    conv0_bias = np.broadcast_to(conv0_bias, (first_ch, first_dim * first_dim))
    conv8_bias = np.broadcast_to(conv8_bias, (125, out_dim * out_dim))
    net_darknet = dn.lib.parse_network_cfg(str(
        QNN_PARAM_DIR / "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg").encode())
    if not net_darknet:
        raise RuntimeError("Darknet failed to load its network configuration.")
    print("Initializing FPGA...", flush=True)
    classifier.init_accelerator()
except BaseException:
    release_model()
    raise
# Only temporary native-library files are placed in RAM.
RESULT_INDEX = 0
detection_history = []
print("Model ready. Intermediate files use RAM and are removed after processing.")

CAPTURE_INDEX = 0


## 3. A modell- és kamerafüggvények definiálása

Futtassa ezt a cellát. Meghatározza a modell szakaszait és a biztonságos képrögzítést. Minden felvételnél a USB-kamera beáll, majd a `finally` blokk felszabadítja, **még a lassú inferencia előtt**. A régi natív könyvtár ideiglenes PNG-fájlt használ a RAM-ban, amelyet az 05. laboratóriummal azonos előfeldolgozás olvas be. Maga a cella még nem nyitja meg a kamerát.

In [ ]:
def load_detector_image(image_path):
    original = None
    letterboxed = None
    try:
        original = dn.load_image(str(image_path).encode(), 0, 0)
        if not original.data:
            raise RuntimeError("Darknet could not read " + str(image_path))
        letterboxed = dn.letterbox_image(original, 416, 416)
        if not letterboxed.data:
            raise RuntimeError("Darknet letterboxing failed.")
        pixels = np.copy(np.ctypeslib.as_array(letterboxed.data, (3, 416, 416)))
        # Preserve the original trained model's paired spatial-axis swaps.
        return np.swapaxes(pixels, 0, 2)[np.newaxis, :, :, :]
    finally:
        if letterboxed is not None and letterboxed.data:
            dn.free_image(letterboxed)
        if original is not None and original.data:
            dn.free_image(original)

def first_convolution(pixels):
    if pixels.shape != (1, 416, 416, 3):
        raise ValueError("Expected one letterboxed 416x416 RGB image.")
    values = utils.conv_layer(pixels, conv0_weights, b=conv0_bias, stride=2, padding=1)
    return utils.quantize(values.clip(0.0, 4.0) / 4.0, 3)

def fpga_layers(quantized):
    if classifier is None:
        raise RuntimeError("Model released. Rerun section 2 before inference.")
    output = classifier.get_accel_buffer(out_ch, out_dim)
    inputs = np.ascontiguousarray(classifier.prepare_buffer(quantized * 7))
    started = time.perf_counter()
    classifier.inference(inputs, output)
    call_seconds = time.perf_counter() - started
    values = classifier.postprocess_buffer(output)
    if values.size != out_ch * out_dim * out_dim or not np.all(np.isfinite(values)):
        raise ValueError("Invalid FPGA output.")
    return values, call_seconds

def last_convolution(values):
    pixels = np.swapaxes(values.reshape(out_dim, out_dim, out_ch), 0, 1)[np.newaxis]
    output = utils.conv_layer(pixels, conv8_weights, b=conv8_bias, stride=1)
    # The native region layer reads float*, so float32 and contiguous storage are required.
    output = np.ascontiguousarray(output, dtype=np.float32)
    if output.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(output)):
        raise ValueError("Invalid detection-head output.")
    return output

def draw_detections(image_path, values, threshold=0.3):
    global RESULT_INDEX
    if net_darknet is None:
        raise RuntimeError("Model released. Rerun section 2.")
    if not 0 < threshold < 1:
        raise ValueError("Detection threshold must be between 0 and 1.")
    # Defensive check also protects callers that rerun this cell independently.
    if values.dtype != np.float32 or not values.flags.c_contiguous:
        raise ValueError("Darknet input must be contiguous float32.")
    if values.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(values)):
        raise ValueError("Wrong detection-head shape or non-finite values.")
    RESULT_INDEX += 1
    result_name = "detection_{:04d}".format(RESULT_INDEX)
    output_prefix = RESULT_DIR / result_name
    output_image = RESULT_DIR / (result_name + ".png")
    probabilities = RESULT_DIR / (result_name + "_probabilities.txt")
    # This legacy C helper uses 256-byte path buffers, including added suffixes.
    paths = [str(image_path), str(VOC_NAMES), str(DARKNET_DIR) + "/",
             str(output_prefix), str(probabilities)]
    if any(len(path.encode()) > 240 for path in paths):
        raise ValueError("Use a shorter notebook/image path for the legacy Darknet helper.")
    for path in (output_image, probabilities):
        if path.exists():
            path.unlink()
    try:
        dn.lib.forward_region_layer_pointer_nolayer(
            net_darknet, values.ctypes.data_as(ctypes.POINTER(ctypes.c_float)))
        dn.lib.draw_detection_python(net_darknet, paths[0].encode(), ctypes.c_float(threshold),
            ctypes.c_float(0.5), paths[1].encode(), paths[2].encode(),
            paths[3].encode(), paths[4].encode())
        if not output_image.is_file() or not probabilities.is_file():
            raise RuntimeError("Darknet did not create the detection outputs.")
        detections = []
        for line in probabilities.read_text().splitlines():
            if not line.strip():
                continue
            name, score = line.rsplit(":", 1)
            detections.append((name.strip(), float(score.strip().rstrip("%")) / 100.0))
        return memory_preview(output_image), sorted(detections, key=lambda item: item[1], reverse=True)
    finally:
        for path in (output_image, probabilities):
            if path.exists():
                path.unlink()


def show_detections(output_image, detections):
    for name, score in detections:
        print("{:20s} {:6.0%}".format(name, score))
    if not detections:
        print("No objects above the selected threshold. This is a valid result.")
    preview = memory_preview(output_image)
    preview.thumbnail((800, 600))
    display(preview)

def capture_frame():
    """Open a board-connected USB camera, get a fresh frame, always close it."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError("Cannot open {}. Connect the USB camera to the board and close other camera kernels.".format(CAMERA_DEVICE))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
        for _ in range(WARMUP_FRAMES):
            ok, frame = cap.read()
            if not ok or frame is None or frame.size == 0:
                raise RuntimeError("No camera frame. Check USB power/connection and camera ownership.")
        return frame.copy()  # OpenCV BGR; cv2.imwrite preserves the correct PNG colors.
    finally:
        cap.release()

def save_capture(frame):
    global CAPTURE_INDEX
    CAPTURE_INDEX += 1
    path = RESULT_DIR / "capture_{:04d}.png".format(CAPTURE_INDEX)
    if not cv2.imwrite(str(path), frame):
        raise IOError("Could not save camera capture: " + str(path))
    return path

def preview_camera():
    frame = capture_frame()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    preview = Image.fromarray(rgb)
    preview.thumbnail((640, 480))
    print("Camera preview — captured now; the camera has been released.")
    display(preview)
    return frame

def run_model(image_path):
    if classifier is None:
        raise RuntimeError("Session finished. Rerun section 2 and the controls cell.")
    if not classifier.init:
        print("Loading YOLO overlay...", flush=True)
        classifier.init_accelerator()
    timings = {}
    started = time.perf_counter()
    pixels = load_detector_image(image_path)
    timings["letterbox_s"] = time.perf_counter() - started
    started = time.perf_counter()
    quantized = first_convolution(pixels)
    timings["first_cpu_s"] = time.perf_counter() - started
    started = time.perf_counter()
    fpga_output, timings["fpga_call_s"] = fpga_layers(quantized)
    timings["fpga_with_buffers_s"] = time.perf_counter() - started
    started = time.perf_counter()
    head = last_convolution(fpga_output)
    timings["last_cpu_s"] = time.perf_counter() - started
    return head, timings

def show_pair(original_path, result_path):
    original = memory_preview(original_path)
    annotated = memory_preview(result_path)
    original.thumbnail((480, 360))
    annotated.thumbnail((480, 360))
    canvas = Image.new("RGB", (original.width + annotated.width, max(original.height, annotated.height) + 28), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 7), "Captured frame", fill="black")
    draw.text((original.width + 8, 7), "Detection result", fill="black")
    canvas.paste(original, (0, 28))
    canvas.paste(annotated, (original.width, 28))
    display(canvas)


## 4. Képrögzítés és felismerés definiálása

Futtassa a cellát a gombművelet meghatározásához; még nem készül felvétel. A felismerés határoló téglalapokat rajzol, és az állóképes példával azonos pontszámokat mutat. A pontszám a modell bizonyosságát, nem a pontosságát jelenti. Az üres találat is érvényes. A mért idők a kamera megnyitását és beállását, a képfájlokat és a modell feldolgozását tartalmazzák; az FPGA-hívás ideje külön jelenik meg. A Jupyter megjelenítése nincs mérve, az eredmény nem élőkamerás FPS.

In [ ]:
def capture_and_detect():
    if classifier is None:
        raise RuntimeError("Session finished. Rerun section 2, then the controls cell.")
    if not 0 < THRESHOLD < 1:
        raise ValueError("THRESHOLD must be between 0 and 1.")
    started = time.perf_counter()
    print("Capturing USB camera...", flush=True)
    try:
        frame = capture_frame()
        capture_path = save_capture(frame)
        print("Captured", capture_path.name, "— processing; please wait...", flush=True)
        head, timings = run_model(capture_path)
        post_started = time.perf_counter()
        result_path, detections = draw_detections(capture_path, head, THRESHOLD)
        timings["decode_draw_s"] = time.perf_counter() - post_started
        timings["capture_to_result_s"] = time.perf_counter() - started
        record = dict(capture=memory_preview(capture_path), result=memory_preview(result_path),
                      threshold=THRESHOLD, detections=detections, timings=timings)
        detection_history.append(record)
        print("Result {} — {}".format(len(detection_history), capture_path.name))
        show_pair(capture_path, result_path)
        for name, score in detections:
            print("{:20s} {:6.0%}".format(name, score))
        if not detections:
            print("No objects above threshold. Try another scene, distance or lighting.")
        print("Capture to result: {:.3f} s; FPGA inference call: {:.3f} s".format(
            timings["capture_to_result_s"], timings["fpga_call_s"]))
        print("Includes camera opening/warmup, model stages, drawing and PNG I/O; excludes notebook rendering.")
        return record
    except BaseException:
        classifier.deinit_accelerator()
        raise
    finally:
        clear_intermediate_files()


def finish_session():
    for control in globals().get("ui_buttons", []):
        control.disabled = True
    release_model()
    print("FPGA runtime and Darknet released. The camera is closed after every capture.")


## 5. Hallgatói kísérletek

Használjon személyt, palackot vagy széket az ismert osztályok közül. Hasonlítson össze három távolságot és két megvilágítási körülményt. Próbáljon részleges takarást. Egy új cellában módosítsa a `THRESHOLD` értékét (például `THRESHOLD = 0.5`), majd készítsen új felvételt. Hasonlítsa össze a megtartott dobozokat és a bizonyossági pontszámokat. Néhány képből nem következtethetünk az adathalmazra érvényes pontosságra. Vizsgálja a CPU-szakaszok és az FPGA-hívás idejét.

## 6. A kamera használata

Egyszer futtassa a vezérlőcellát, majd használja a gombokat:

1. **Preview camera** gyors állóképes előnézetet készít a kamera beállításához. Nem futtat inferenciát, és nem élő videó.
2. **Capture + detect** kattintáskor új képet készít és feldolgozza. A rögzítés pillanatában az objektum maradjon látható, majd várja meg a **Ready** állapotot. Az eredeti és a feldolgozott kép egymás mellett jelenik meg, utánuk a pontszámok és az idők. A jelenet az előnézet óta változhatott.
3. Ismételje a kívánt alkalommal. Az eredmények hozzáadódnak az előzményekhez. A **Finish / release** kikapcsolja a vezérlőket és felszabadítja a modellt.

Feldolgozás közben a gombok le vannak tiltva az átfedő kamera/FPGA-hívások elkerülésére. Kézi módban futtassa a `preview_camera()` vagy `capture_and_detect()` függvényt új cellában. Egy Jupyter-cella ismételt futtatása saját kimenetét cseréli le; a memóriabeli előzmények a kernel újraindításáig elérhetők.

In [ ]:
# Close old controls so a rerun cannot leave a second set of active buttons.
for control in globals().get("ui_buttons", []):
    control.disabled = True
if globals().get("ui_panel") is not None:
    ui_panel.close()

if widgets is not None:
    preview_button = widgets.Button(description="Preview camera", icon="camera")
    detect_button = widgets.Button(description="Capture + detect", icon="search", button_style="primary")
    finish_button = widgets.Button(description="Finish / release", icon="stop", button_style="warning")
    status = widgets.HTML("Ready. Aim the camera, optionally preview, then capture and detect.")
    history_output = widgets.Output()
    ui_buttons = [preview_button, detect_button, finish_button]
    ui_panel = widgets.VBox([widgets.HBox(ui_buttons), status, history_output])

    def perform(action):
        for control in ui_buttons:
            control.disabled = True
        status.value = "Working — wait for the current capture/inference to finish."
        try:
            with history_output:
                action()
            status.value = "Ready. Previous images remain below."
        except Exception:
            with history_output:
                traceback.print_exc()
            status.value = "Operation failed. See the message below; previous results are retained."
        finally:
            for control in ui_buttons:
                control.disabled = classifier is None

    def finish_clicked(_):
        for control in ui_buttons:
            control.disabled = True
        finish_session()
        status.value = "Session released. Rerun section 2 and this cell to start again."

    preview_button.on_click(lambda _: perform(preview_camera))
    detect_button.on_click(lambda _: perform(capture_and_detect))
    finish_button.on_click(finish_clicked)
    display(ui_panel)
else:
    ui_buttons = []
    ui_panel = None
    print("Manual mode: run preview_camera() or capture_and_detect() in a new cell.")


## 7. A memóriabeli munkamenet eredményeinek újramegjelenítése (opcionális)

A cella együtt jeleníti meg az összes befejezett kép/detekció párt, a kézi hívások eredményeivel együtt. A memóriából olvas, **nem** futtat új inferenciát.

In [ ]:
print("Completed detections:", len(detection_history))
for number, record in enumerate(detection_history, 1):
    print("Result", number)
    show_pair(record["capture"], record["result"])



## 8. Befejezés és memóriafelszabadítás

Használja a **Finish / release** gombot, vagy futtassa a következő cellát. Mindkettő biztonságosan ismételhető. Másik FPGA-notebook előtt állítsa le ezt a kernelt. Újrakezdéshez futtassa a 2. szakaszt és a vezérlőcellát; a megváltoztatott beállításokat is állítsa vissza.

Hosszú natív hívásnál a **Kernel → Interrupt** csak a hívás visszatérésekor hathat. Ne futtasson párhuzamosan második FPGA-notebookot.

In [ ]:
finish_session()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Large weights cleared. No capture or result files were saved.")


Megvalósítási hivatkozások: [OpenCV VideoCapture](https://docs.opencv.org/3.4.20/d8/dfe/classcv_1_1VideoCapture.html), [QNN-MO-PYNQ](https://github.com/onigas/QNN-MO-PYNQ), [Darknet region layer](https://github.com/giuliogamba/darknet/blob/master/src/region_layer.c) és [PYNQ-Z2 base overlay](https://pynq.readthedocs.io/en/v2.5/pynq_overlays/pynqz2/pynqz2_base_overlay.html).

**Validálás:** a notebook szintaxisa, a kamera/FPGA erőforrás-kezelése, az ismételt inicializálás, a memóriabeli előzmények, a callback-felszabadítás és a Python-folyamat off-board ellenőrzése szimulált natív/hardver interfészekkel történt. A valódi USB-kamerát, widgeteket, Darknetet, FPGA-időket/predikciókat és a fizikai LED működését még PYNQ-hardveren kell ellenőrizni.